In [1]:
import os

dataset_path = "data/train"

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        print(f"{class_name}: {count} images")

battery: 755 images
biological: 797 images
cardboard: 1460 images
clothes: 4261 images
glass: 2448 images
metal: 816 images
paper: 1344 images
plastic: 1587 images
shoes: 1581 images
trash: 757 images


## Étape 1+2 — Préparation Dataset + Class Weights + DataLoader

In [2]:
# ===== Import Libraries =====
import os
import torch
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# ===== Dataset Paths =====
train_dir = "data/train"
val_dir = "data/val"

# ===== Classes & Counts =====
classes = [
    "battery", "biological", "cardboard", "clothes",
    "glass", "metal", "paper", "plastic", "shoes", "trash"
]

counts = [755, 797, 1460, 4261, 2448, 816, 1344, 1587, 1581, 757]

# ===== Compute Class Weights =====
labels = []
for i, count in enumerate(counts):
    labels += [i] * count

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

weights = torch.tensor(class_weights, dtype=torch.float)
print("Class Weights:", weights)

# ===== Data Transformations =====
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# ===== Datasets =====
train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
val_dataset = ImageFolder(root=val_dir, transform=val_transforms)

# ===== DataLoaders =====
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ===== Quick Check =====
print("Number of training samples:", len(train_dataset))
print("Number of validation samples:", len(val_dataset))
print("Classes:", train_dataset.classes)

Class Weights: tensor([2.0935, 1.9832, 1.0826, 0.3709, 0.6457, 1.9370, 1.1760, 0.9960, 0.9997,
        2.0880])
Number of training samples: 15806
Number of validation samples: 3956
Classes: ['battery', 'biological', 'cardboard', 'clothes', 'glass', 'metal', 'paper', 'plastic', 'shoes', 'trash']


## Étape 3+4 — Training + MLflow

In [3]:
# ===== Imports =====
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm
import mlflow
import mlflow.pytorch

# ===== Device =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===== Load Pretrained EfficientNet B0 =====
model = models.efficientnet_b0(pretrained=True)

# Freeze feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier
num_classes = 10  # battery, biological, ...
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, num_classes)
model = model.to(device)

# ===== Loss + Optimizer =====
criterion = nn.CrossEntropyLoss(weight=weights.to(device))  
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ===== MLflow Setup =====
mlflow.set_experiment("Trash_Classification")

# ===== Training Loop =====
num_epochs = 6

with mlflow.start_run():

    # Log parameters
    mlflow.log_param("model", "efficientnet_b0")
    mlflow.log_param("epochs", num_epochs)
    mlflow.log_param("lr", 0.001)
    mlflow.log_param("batch_size", 32)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / total
        epoch_acc = correct / total

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_epoch_loss = val_loss / val_total
        val_epoch_acc = val_correct / val_total

        print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f}, Acc={epoch_acc:.4f} | Val Loss={val_epoch_loss:.4f}, Acc={val_epoch_acc:.4f}")

        # Log metrics to MLflow
        mlflow.log_metric("train_loss", epoch_loss, step=epoch+1)
        mlflow.log_metric("train_acc", epoch_acc, step=epoch+1)
        mlflow.log_metric("val_loss", val_epoch_loss, step=epoch+1)
        mlflow.log_metric("val_acc", val_epoch_acc, step=epoch+1)

    # Save the final model in MLflow
    mlflow.pytorch.log_model(model, "efficientnet_model")
    print("Model saved in MLflow ")

c:\Users\HP\Desktop\Ing4\Projetpython-application-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


c:\Users\HP\Desktop\Ing4\Projetpython-application-\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\HP\Desktop\Ing4\Projetpython-application-\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/6: 100%|██████████| 494/494 [12:45<00:00,  1.55s/it]


Epoch 1: Train Loss=0.8846, Acc=0.7738 | Val Loss=0.4906, Acc=0.8612


Epoch 2/6: 100%|██████████| 494/494 [11:03<00:00,  1.34s/it]


...

Epoch 5: Train Loss=0.4824, Acc=0.8529 | Val Loss=0.3779, Acc=0.8870


Epoch 6/6: 100%|██████████| 494/494 [11:50<00:00,  1.44s/it]


Epoch 6: Train Loss=0.4861, Acc=0.8523 | Val Loss=0.3745, Acc=0.8870


2026/02/22 01:46:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 01:46:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Model saved in MLflow 


In [4]:
mlflow.pytorch.log_model(model, name="efficientnet_model")
print("Model saved in MLflow ")

2026/02/22 01:56:47 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Model saved in MLflow 
